In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# PHASE 2: CLEANING 

In [ ]:
df = pd.read_csv('yahoo_finance.csv')

In [ ]:
df.head()

In [ ]:
# 1. Handle Missing Values & Duplicates
df_clean = df.drop_duplicates().reset_index(drop=True)
df_clean['PriceCategory'] = pd.cut(df_clean['Price'], bins=[0, 20, 100, 1000], labels=['Low', 'Medium', 'High'])
df_clean.to_csv('cleaned_yahoo_finance.csv', index=False)
print('cleaned_yahoo_finance.csv')

In [ ]:
# 2. Fix inconsistent formats
# Ensure Price and MarketCap are numeric
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
df['MarketCap'] = pd.to_numeric(df['MarketCap'], errors='coerce')

In [ ]:
# 3. Feature Engineering: Create useful features
# Price Band (Low, Mid, High)
df['Price_Band'] = pd.qcut(df['Price'], 3, labels=["Low", "Mid", "High"])

# Market Cap Category
df['Size'] = pd.cut(df['MarketCap'], 
                    bins=[0, 2e9, 10e9, 100e9, np.inf], 
                    labels=['Small-Cap', 'Mid-Cap', 'Large-Cap', 'Mega-Cap'])

# Volatility Indicator (Absolute change relative to price)
df['Volatility_Index'] = (df['Change'].abs() / df['Price']) * 100

print("Cleaned Dataset Preview:")
print(df.head())

In [ ]:
# Save for next phase
df.to_csv('cleaned_yahoo_finance.csv', index=False)

# PHASE 3: EDA

In [ ]:
 # 1. Summary Statistics
print(df.describe())

In [ ]:
# 2. Plotting Top 10 Market Cap
top_10 = df_clean.sort_values(by='MarketCap', ascending=False).head(10)
sns.barplot(data=top_10, x='MarketCap', y='Company')
plt.title('Top 10 Companies by Market Cap')
plt.savefig('market_cap.png')

In [ ]:
# 3. Visualizations
plt.figure(figsize=(10, 5))

# Subplot 1: Distribution of Price Bands
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='Price_Band', palette='viridis')
plt.title('Distribution of Price Bands')

# Subplot 2: Correlation Heatmap
plt.subplot(1, 2, 2)
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')

plt.tight_layout()
plt.show()

In [ ]:
# 4. Outlier Detection (Boxplot for Volume)
plt.figure(figsize=(8, 4))
sns.boxplot(x=df['Volume'])
plt.title('Trading Volume Outliers')
plt.show()

# PHASE 4: MACHINE LEARNING

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# 1. Select Features and Target
X = df[['Volume', 'Change', 'MarketCap']]
y = df['Price']

In [ ]:
# 2. Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 3. Model Training
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
# 4. Evaluation
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Model Performance:")
print(f"RMSE (Root Mean Squared Error): {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

In [ ]:
# 5. Output Prediction Comparison
results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
print(results.head())